# 4.13 · 梯度提升回归 / Gradient Boosting (GBDT & XGBoost)

> **课程定位 / Where this fits**
> 第 13 课，**Part 4 · 监督学习：回归**。
> Lesson 13, **Part 4 · Supervised Regression**.
>
> 随机森林(4.12)是 **bagging**（并行、独立、降方差）。GBDT 是 **boosting**（串行、每棵树纠正前面的残差、降偏差）。它是**表格数据之王**——Kaggle 表格赛常年由 XGBoost/LightGBM 霸榜。这一课从零实现 GBDT 的核心一行（拟合残差），再上 XGBoost 与早停。(分类版见 5.8-5.11。)
> Random Forest (4.12) is **bagging** (parallel, independent, variance reduction). GBDT is **boosting** (sequential, each tree corrects the residual, bias reduction). It's the **king of tabular data** — Kaggle tabular leaderboards are dominated by XGBoost/LightGBM. We implement GBDT's core line (fit residuals) from scratch, then move to XGBoost with early stopping. (Classification in 5.8-5.11.)
>
> 💼 **实战/面试视角**："boosting 怎么工作 / GBDT vs RF / 学习率和早停 / XGBoost 为什么强" 是表格建模**最核心**的题。
> 💼 **Practical/interview angle:** "how boosting works / GBDT vs RF / learning rate & early stopping / why XGBoost wins" are *the* core tabular questions.

> 💡 **面试相关 / Interview-relevant**
> - "GBDT 怎么工作（每棵树拟合残差/负梯度）"（出镜率 ★★★★★）
> - "GBDT vs 随机森林"（★★★★★，串行降偏差 vs 并行降方差）
> - "学习率的作用 / 为什么需要早停"（★★★★★）
> - "GBDT 为什么会过拟合（树太多）"（★★★★★）
> - "XGBoost 比普通 GBDT 强在哪"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 boosting = 一棵接一棵拟合残差（平方损失下负梯度=残差）。
   Understand boosting = sequentially fitting residuals (negative gradient = residual under squared loss).
2. **从零**实现 GBDT 核心，对照 sklearn。
   Implement GBDT's core from scratch, matching sklearn.
3. 理解学习率 × 树数的权衡，用**早停**防过拟合。
   Understand the learning-rate × tree-count trade-off; use **early stopping**.
4. 用 **XGBoost** + 早停 + 正则。
   Use XGBoost with early stopping and regularization.
5. 横向对比单树/森林/GBDT/XGBoost。
   Compare tree/forest/GBDT/XGBoost head-to-head.

## 目录 / TOC
1. [先建直觉：串行纠错 ⭐](#1)
2. [从零 GBDT：拟合残差 ⭐](#2)
3. [boosting 逐步纠错（可视化）⭐](#3)
4. [学习率 + 早停 ⭐](#4)
5. [XGBoost + 早停 ⭐](#5)
6. [四模型横向对比 + 小结 ⭐](#6)


<a id="1"></a>
## 1. 先建直觉：串行纠错 ⭐ / Intuition: Sequential Correction

随机森林是"很多专家**同时**投票"。boosting 换了个思路：**一个接一个地请专家，每个新专家专门补前面所有人留下的错。**
Random Forest is "many experts voting **at once**". Boosting takes a different tack: **bring in experts one at a time, each correcting the errors the others left behind.**

具体到回归：先用一个简单预测（如全局均值），算出每个点的**残差**（真实 − 当前预测）；再训一棵浅树**专门拟合这些残差**；把它（乘一个小学习率）加进来；重新算残差，再训下一棵……如此反复。每棵树都在"补前面的差"，无数弱树叠成强模型。
For regression: start with a simple prediction (e.g. the global mean), compute each point's **residual** (actual − current prediction); train a shallow tree to **fit those residuals**; add it in (scaled by a small learning rate); recompute residuals; repeat. Each tree "patches the leftover error", and many weak trees compound into a strong model.

**为什么叫"梯度"提升**：在平方损失下，残差 $(y - F)$ 恰好是损失对预测 $F$ 的**负梯度**。所以"拟合残差"其实是"在函数空间做梯度下降"，换其它损失就拟合对应的负梯度（5.8 详述）。
**Why "gradient" boosting:** under squared loss, the residual $(y - F)$ is exactly the **negative gradient** of the loss w.r.t. the prediction $F$. So "fitting residuals" is "gradient descent in function space"; other losses fit their own negative gradients (detailed in 5.8).


<a id="2"></a>
## 2. 从零 GBDT：拟合残差 ⭐ / From Scratch: Fit Residuals

GBDT 的核心代码出奇地短——整个算法就是循环里那两行：**算残差 → 训一棵树拟合残差 → 累加**。继续用 Diamonds。
GBDT's core code is surprisingly short — the whole algorithm is two lines in the loop: **compute residual → train a tree on it → accumulate**. Continuing with Diamonds.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(8000, random_state=0).reset_index(drop=True)
feat_names = ["carat","depth","table","x","y","z"]
X = df[feat_names].values; y = df["price"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

class MyGBDT:
    def __init__(self, n_estimators=100, lr=0.1, max_depth=3):
        self.n_estimators, self.lr, self.max_depth = n_estimators, lr, max_depth
    def fit(self, X, y):
        self.F0 = y.mean()                    # 初始预测 = 全局均值 / start from the mean
        F = np.full(len(y), self.F0)          # F = 每个样本的当前预测
        self.trees = []
        for _ in range(self.n_estimators):
            residual = y - F                  # 残差 = 真实-当前预测 (平方损失的负梯度)
            tree = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, residual)  # 树拟合残差
            F += self.lr * tree.predict(X)    # 把 lr×新树 的修正累加进当前预测
            self.trees.append(tree)
        return self
    def predict(self, X):
        return self.F0 + self.lr * sum(t.predict(X) for t in self.trees)   # 初值 + 所有树累加

my = MyGBDT(n_estimators=100, lr=0.1, max_depth=3).fit(X_tr, y_tr)
print(f"从零 GBDT test R² = {r2_score(y_te, my.predict(X_te)):.4f}")
sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr)
print(f"sklearn GBDT  test R² = {sk.score(X_te, y_te):.4f}")
print("→ 核心就是'每棵树拟合当前残差'这一行; 从零与库结果接近")


<a id="3"></a>
## 3. boosting 逐步纠错（可视化）⭐ / Boosting Corrects Step by Step

在单特征上画出"加了 1 棵 / 10 棵 / 100 棵树之后"的预测：1 棵树是粗糙的阶梯，随着树越加越多，预测逐步细化、贴合数据。这就是串行纠错的累积效果——和随机森林"一次性平均"的画风完全不同。
On one feature, plot the prediction "after 1 / 10 / 100 trees": one tree is a coarse staircase; as more trees accumulate, the prediction refines and hugs the data. That's the cumulative effect of sequential correction — very different from RF's "average all at once".


In [ ]:
xc = df["carat"].values.reshape(-1,1); yc = y
order = np.argsort(xc.ravel()); xc, yc = xc[order], yc[order]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, [1, 10, 100]):
    g = GradientBoostingRegressor(n_estimators=n, learning_rate=0.3, max_depth=2, random_state=0).fit(xc, yc)
    ax.scatter(xc, yc, alpha=0.08, s=6); ax.plot(xc, g.predict(xc), "r-", lw=2)
    ax.set_title(f"{n} 棵树后 after {n} trees")
plt.tight_layout(); plt.show()
print("1 棵: 粗糙阶梯; 10 棵: 渐细; 100 棵: 平滑贴合 — 逐步纠错的累积效果")


<a id="4"></a>
## 4. 学习率 + 早停 ⭐ / Learning Rate & Early Stopping

**学习率(learning_rate)**：每棵树只迈一小步（把它的修正缩小）。小学习率 → 需要更多树，但泛化更好；大学习率 → 快但易过拟合。
**learning_rate:** each tree takes a small step (its correction is shrunk). Small LR → needs more trees but generalizes better; large LR → fast but overfit-prone.

**GBDT 会过拟合，且树越多越严重**（与随机森林相反！）——因为每棵新树都在更努力地拟合训练集。所以必须**早停(early stopping)**：用验证集监控，当验证误差不再下降就停。`staged_predict` 能让我们看到每一步的 train/test 误差，清楚看到"train 一直降、test 先降后升"的过拟合拐点。
**GBDT overfits, and more trees makes it worse** (opposite of RF!) — each new tree fits the training set harder. So you must use **early stopping**: monitor a validation set and stop when validation error stops improving. `staged_predict` lets us watch per-step train/test error and see the overfitting turning point where "train keeps dropping but test rises".


In [ ]:
# 学习率影响 / learning-rate effect
for lr in [0.01, 0.1, 0.5, 1.0]:
    g = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, max_depth=3, random_state=0).fit(X_tr, y_tr)
    tag = "(大lr: train高test低=过拟合迹象)" if lr >= 0.5 else ""
    print(f"lr={lr:<5} train R²={g.score(X_tr,y_tr):.4f}  test R²={g.score(X_te,y_te):.4f}  {tag}")
print("小 lr 泛化好但需更多树; 大 lr 快但易过拟合(train-test 缺口大)\n")

# 用 staged_predict 追踪每棵树后的 train/test MSE → 看过拟合拐点 / per-stage error
g = GradientBoostingRegressor(n_estimators=500, learning_rate=0.1, max_depth=4, random_state=0).fit(X_tr, y_tr)
train_err = [mean_squared_error(y_tr, p) for p in g.staged_predict(X_tr)]   # 每加一棵树的 train MSE
test_err  = [mean_squared_error(y_te, p) for p in g.staged_predict(X_te)]
best_iter = int(np.argmin(test_err)) + 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_err, label="train MSE"); ax.plot(test_err, label="test MSE")
ax.axvline(best_iter, color="r", ls="--", label=f"最优树数={best_iter} (早停点)")
ax.set_xlabel("树数(boosting 轮)"); ax.set_ylabel("MSE"); ax.legend()
ax.set_title("GBDT: train 一直降, 但 test 先降后升(过拟合) → early stopping")
plt.tight_layout(); plt.show()
print(f"train MSE 单调降; test MSE 在 ~{best_iter} 棵后回升=过拟合 → 早停在拐点停")


<a id="5"></a>
## 5. XGBoost + 早停 ⭐ / XGBoost with Early Stopping

**XGBoost** 在普通 GBDT 上加了一堆改进（数学+工程）：二阶泰勒近似的目标、叶子权重的 L1/L2 正则、列采样、并行、原生早停等。结果是更准、更快、更不易过拟合——这就是它霸榜 Kaggle 的原因。(原理在 5.9 详述。)
**XGBoost** adds many improvements (math + engineering) over plain GBDT: a second-order Taylor objective, L1/L2 regularization on leaf weights, column subsampling, parallelism, native early stopping. The result is more accurate, faster, more overfit-resistant — why it dominates Kaggle. (Details in 5.9.)


In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.1, max_depth=4,
    reg_lambda=1.0, reg_alpha=0.0,          # L2 / L1 正则(叶子权重)
    early_stopping_rounds=20, random_state=0, n_jobs=-1,
)
# eval_set 提供验证集, 连续 20 轮不提升就停 / early stopping on validation set
xgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print(f"XGBoost test R² = {xgb_model.score(X_te, y_te):.4f}")
print(f"早停在第 {xgb_model.best_iteration} 棵树停止(而非全部 500)")

imp = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.2))
imp.plot(kind="barh", ax=ax); ax.invert_yaxis(); ax.set_title("XGBoost 特征重要性")
plt.tight_layout(); plt.show()
print("💡 LightGBM(leaf-wise 生长) 和 CatBoost(原生类别+有序提升) 是 XGBoost 的强力同类")
print("   三者在 Part 5.8-5.11 分类场景详细对比")


<a id="6"></a>
## 6. 四模型横向对比 + 小结 ⭐ / Four Models Compared & Summary

在同一份 Diamonds 上比单树 / 随机森林 / sklearn GBDT / XGBoost 的精度和训练时间。典型结论：**单树 < 森林 < GBDT ≈ XGBoost**，XGBoost 通常精度最高且训练快——这就是表格赛它常年霸榜的原因。
On the same Diamonds data, compare tree / RF / sklearn GBDT / XGBoost on accuracy and training time. Typical conclusion: **tree < forest < GBDT ≈ XGBoost**, with XGBoost usually most accurate and fast — why it tops tabular leaderboards.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
import time

contestants = {
    "单棵树 tree":     DecisionTreeRegressor(max_depth=10, random_state=0),
    "随机森林 RF":      RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "sklearn GBDT":    GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=0),
    "XGBoost":         xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=0, n_jobs=-1),
}
print(f"{'模型 model':<18} {'test R²':>9} {'训练秒 sec':>10}")
for name, m in contestants.items():
    t0 = time.perf_counter(); m.fit(X_tr, y_tr); dt = time.perf_counter() - t0
    print(f"{name:<18} {m.score(X_te, y_te):>9.4f} {dt:>10.2f}")
print("\n精度: 单树 < 森林 < GBDT ≈ XGBoost; XGBoost 通常精度最高且训练快")
print("这就是 Kaggle 表格赛 XGBoost/LightGBM 常年霸榜的原因")


```
boosting: 串行, 每棵浅树拟合前面的残差(平方损失下 残差=负梯度), 降偏差
  核心一行: residual = y - F; tree.fit(X, residual); F += lr × tree.predict(X)
vs 随机森林: RF 并行降方差(树越多越安全); GBDT 串行降偏差(树太多会过拟合!)
学习率: 小 lr 泛化好需更多树, 大 lr 快易过拟合; 配早停(验证误差不降就停)
XGBoost: 二阶泰勒+叶子正则(L1/L2)+列采样+并行+原生早停 → 更准更快(原理 5.9)
精度梯队: 单树 < 森林 < GBDT ≈ XGBoost; 表格数据之王
```

### 💡 面试速查 / Interview cheat-sheet
1. **GBDT 每棵树拟合残差**(平方损失下残差=负梯度); 串行降偏差。
   Each GBDT tree fits the residual (= negative gradient under squared loss); sequential bias reduction.
2. **GBDT vs RF**: 串行降偏差 vs 并行降方差; **GBDT 树太多会过拟合, RF 不会**。
   GBDT vs RF: sequential bias vs parallel variance; GBDT overfits with too many trees, RF doesn't.
3. **小学习率 + 多树 + 早停**是标准配方。
   Small LR + many trees + early stopping is the standard recipe.
4. **XGBoost 强在**: 二阶目标 + 叶子正则 + 列采样 + 并行 + 早停。
   XGBoost wins via second-order objective + leaf regularization + column subsampling + parallelism + early stopping.
5. **树类不需缩放**; 表格数据 XGBoost/LightGBM 是首选。
   Tree-based models need no scaling; XGBoost/LightGBM are the go-to for tabular.

### 下一节 / Next
**4.14 分位数回归**——前面都预测"均值"。分位数回归预测**条件分位数**(如中位数、90 分位), 用于预测区间和对异常值稳健的场景。
**4.14 Quantile Regression** — so far we predicted the mean. Quantile regression predicts conditional quantiles (median, 90th percentile) for prediction intervals and outlier-robust settings.
